### Preprocessing
1) Loads both validation and test datasets
2) Checks the entity distribution 
3) Imputes emails into the datasets
4) Saves the new modified datasets in the "datasets" folder 

In [ ]:
import json
import random
from pathlib import Path


DOMAINS = ['gmail.com', 'yahoo.com', 'outlook.com', 'live.com', 'hotmail.com']

SEPARATORS = ['-', '_', '.', '']


def load_data(filepath: Path):
    print(f"Loading {filepath}")

    with open(filepath, 'r', encoding='utf-8') as file:
        data = json.load(file)

    print(f"File loaded with {len(data)} entries.")
    return data


def analyze_distribution(data):
    """ Checks the tag distributions in the dataset """
    counts = {}
    for entry in data:
        for tag in entry['ner_tags']:
            if tag not in counts:
                counts[tag] = 0
            counts[tag] += 1
    print()
    print("\033[1mTag Distribution\033[0m")
    total = sum(counts.values())

    for tag, count in counts.items():
        print(f"{tag} : {count} ({count / total * 100 :.2f}%)")

    if 'B-EMAIL' not in counts:
        print("No emails in dataset, imputation required.")


def extract_names(data):
    """ Extracts names from the given dataset and returns two lists, first_names and last_names """

    first_names = set()
    last_names = set()
    ignore = {'is', 'poor', 'a', 'the', 'that'}

    for entry in data:
        tags = entry['ner_tags']
        tokens = entry['tokens']

        current_name = []
        for token, tag in zip(tokens, tags):
            token = token.lower()
            if tag == 'B-PER':
                current_name = [token]

            elif tag == 'I-PER':
                current_name.append(token)

            else:
                if current_name:
                    if current_name[0] not in ignore:
                        first_names.add(current_name[0])
                        for name in current_name[1:]:
                            if name not in ignore:
                                last_names.add(name)
                current_name = []

        if current_name:
            if current_name[0] not in ignore:
                first_names.add(current_name[0])
                for name in current_name[1:]:
                    if name not in ignore:
                        last_names.add(name)

    return list(first_names), list(last_names)                      



def generate_email(first_names, last_names):
    """ Generates a realistic synthetic email address """

    first = random.choice(first_names)
    last  = random.choice(last_names)
    sep   = random.choice(SEPARATORS)
    domain = random.choice(DOMAINS)
    randnum = random.randint(1, 999)
    pattern = random.choice([
        f"{first}{sep}{last}",
        f"{first[0]}{sep}{last}",
        f"{first}{sep}{last[0]}",
        f"{first}",
        f"{first}{randnum}",
        f"{last}{randnum}",
        f"{first}{sep}{last}{randnum}",
        f"{first}{randnum}{last}",
        f"{last}{first}", 
        f"{last}{sep}{first}",
        f"{first[0]}{last}{randnum}",       
        f"{first}{sep}{last[0]}{randnum}"
    ])
 
    return f"{pattern}@{domain}"



def main():
    data = load_data('../datasets/data.json')
    analyze_distribution(data)
    print()
    test_data = load_data('../datasets/test_data.json')
    analyze_distribution(test_data)
    first_names, last_names = extract_names(data)
    print(f"\nGenerated synthetic email example: {generate_email(first_names, last_names)}")

if __name__ == '__main__':
    main()

Loading ../datasets/data.json
File loaded with 28516 entries.

Tag Distribution
O : 639055 (90.16%)
B-PER : 40264 (5.68%)
I-PER : 29466 (4.16%)
No emails in dataset, imputation required.

Loading ../datasets/test_data.json
File loaded with 3650 entries.

Tag Distribution
O : 77866 (89.47%)
B-PER : 5207 (5.98%)
I-PER : 3959 (4.55%)
No emails in dataset, imputation required.

Generated synthetic email example: basashok@gmail.com
